# Baseline: PCA + k-nearest neighbours

PCA (100 components) on standardized train+validation data, then KNN for k = 1..14 on each of the six label sets. Accuracy is measured on the held-out test set.

In [ ]:
from pathlib import Path

# Folder holding the .tsv files from https://zenodo.org/records/4326602
# (moanna_training_data.tsv, moanna_training_label.tsv, moanna_validation_*.tsv, moanna_testing_*.tsv)
DATA_DIR = Path("../data")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import preprocessing

In [ ]:
dat_df = pd.read_csv(DATA_DIR / 'moanna_training_data.tsv', sep="\t", index_col=0)
lab_df = pd.read_csv(DATA_DIR / 'moanna_training_label.tsv', sep="\t", index_col=0)

dat_df_val = pd.read_csv(DATA_DIR / 'moanna_validation_data.tsv', sep="\t", index_col=0)
lab_df_val = pd.read_csv(DATA_DIR / 'moanna_validation_label.tsv', sep="\t", index_col=0)

dat_df_test = pd.read_csv(DATA_DIR / 'moanna_testing_data.tsv', sep="\t", index_col=0)
lab_df_test = pd.read_csv(DATA_DIR / 'moanna_testing_label.tsv', sep="\t", index_col=0)

In [ ]:
label_df_concated = pd.concat([lab_df, lab_df_val])
label_df_concated

In [ ]:
data_df_concated = pd.concat([dat_df, dat_df_val])
data_df_concated.shape

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Fit scaler and PCA on the training data only, then apply the same transform to the test set.
scaler = StandardScaler()
df_standardized = scaler.fit_transform(data_df_concated)
df_standardized_test = scaler.transform(dat_df_test)

pca = PCA(n_components=100)
data_df_with_PCA = pca.fit_transform(df_standardized)
data_df_with_PCA_test = pca.transform(df_standardized_test)

In [ ]:
data_df_with_PCA.shape

In [ ]:
data_df_with_PCA_test.shape

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from sklearn import metrics


In [ ]:




ERStatus_train = label_df_concated['ERStatus'].values
HER2Status_train = label_df_concated['HER2Status'].values
Pam50Subtype_train = label_df_concated['Pam50Subtype'].values
Pam50SubtypeSingleLuminal_train = label_df_concated['Pam50SubtypeSingleLuminal'].values
BasalNonBasal_train = label_df_concated['BasalNonBasal'].values
Luminal_train = label_df_concated['Luminal'].values


list_all_label_train = [ERStatus_train, HER2Status_train,
                        Pam50Subtype_train, Pam50SubtypeSingleLuminal_train,
                        BasalNonBasal_train, Luminal_train]



ERStatus_test = lab_df_test['ERStatus'].values
HER2Status_test = lab_df_test['HER2Status'].values
Pam50Subtype_test = lab_df_test['Pam50Subtype'].values
Pam50SubtypeSingleLuminal_test = lab_df_test['Pam50SubtypeSingleLuminal'].values
BasalNonBasal_test = lab_df_test['BasalNonBasal'].values
Luminal_test = lab_df_test['Luminal'].values


list_all_label_test = [ERStatus_test, HER2Status_test,
                      Pam50Subtype_test, Pam50SubtypeSingleLuminal_test,
                      BasalNonBasal_test, Luminal_test]
res_list = []
for k in range(1,15):
    res_list_for_each_k = []
    for i in range(len(list_all_label_train)):
        KNN_Obj = KNeighborsClassifier(n_neighbors = k).fit(data_df_with_PCA,list_all_label_train[i])
        Acc = metrics.accuracy_score(list_all_label_test[i], KNN_Obj.predict(data_df_with_PCA_test))
        res_list_for_each_k.append(Acc)
    res_list.append(res_list_for_each_k)


In [ ]:
for i in res_list:
    print(sum(i)/len(i))